# 02 — Transform Pattern

DataFrame -> DataFrame reusable steps

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("transform-pattern")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

In [2]:
schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("event_time_raw", StringType(), True),
])

rows = [
    ("e1","u1","Purchase",10.0,"2026-01-01 10:00:00"),
    ("e2",None,"purchase",20.0,"2026-01-01 10:05:00"),
    ("e3","u2","refund",5.0,"2026-01-01 10:10:00"),
    ("e4","u3","purchase",-1.0,"2026-01-01 10:15:00"),
]

df = spark.createDataFrame(rows, schema)
df.show()

[Stage 0:>                                                          (0 + 1) / 1]

+--------+-------+----------+------+-------------------+
|event_id|user_id|event_type|amount|     event_time_raw|
+--------+-------+----------+------+-------------------+
|      e1|     u1|  Purchase|  10.0|2026-01-01 10:00:00|
|      e2|   NULL|  purchase|  20.0|2026-01-01 10:05:00|
|      e3|     u2|    refund|   5.0|2026-01-01 10:10:00|
|      e4|     u3|  purchase|  -1.0|2026-01-01 10:15:00|
+--------+-------+----------+------+-------------------+



In [3]:
def normalize(df):
    return df.withColumn("event_type", F.lower(F.col("event_type")))

def parse_time(df):
    return df.withColumn("event_time", F.to_timestamp("event_time_raw"))

def validate(df):
    return (
        df
        .withColumn("error_reason",
            F.when(F.col("user_id").isNull(), F.lit("missing_user"))
             .when(F.col("amount") < 0, F.lit("negative_amount"))
        )
        .withColumn("is_valid", F.col("error_reason").isNull())
    )

In [4]:
prepared = (
    df
    .transform(normalize)
    .transform(parse_time)
    .transform(validate)
)

prepared.show()

[Stage 2:>                                                          (0 + 1) / 1]

+--------+-------+----------+------+-------------------+-------------------+---------------+--------+
|event_id|user_id|event_type|amount|     event_time_raw|         event_time|   error_reason|is_valid|
+--------+-------+----------+------+-------------------+-------------------+---------------+--------+
|      e1|     u1|  purchase|  10.0|2026-01-01 10:00:00|2026-01-01 10:00:00|           NULL|    true|
|      e2|   NULL|  purchase|  20.0|2026-01-01 10:05:00|2026-01-01 10:05:00|   missing_user|   false|
|      e3|     u2|    refund|   5.0|2026-01-01 10:10:00|2026-01-01 10:10:00|           NULL|    true|
|      e4|     u3|  purchase|  -1.0|2026-01-01 10:15:00|2026-01-01 10:15:00|negative_amount|   false|
+--------+-------+----------+------+-------------------+-------------------+---------------+--------+



In [5]:
silver = prepared.filter("is_valid")
quarantine = prepared.filter("NOT is_valid")

silver.show()
quarantine.show()

+--------+-------+----------+------+-------------------+-------------------+------------+--------+
|event_id|user_id|event_type|amount|     event_time_raw|         event_time|error_reason|is_valid|
+--------+-------+----------+------+-------------------+-------------------+------------+--------+
|      e1|     u1|  purchase|  10.0|2026-01-01 10:00:00|2026-01-01 10:00:00|        NULL|    true|
|      e3|     u2|    refund|   5.0|2026-01-01 10:10:00|2026-01-01 10:10:00|        NULL|    true|
+--------+-------+----------+------+-------------------+-------------------+------------+--------+

+--------+-------+----------+------+-------------------+-------------------+---------------+--------+
|event_id|user_id|event_type|amount|     event_time_raw|         event_time|   error_reason|is_valid|
+--------+-------+----------+------+-------------------+-------------------+---------------+--------+
|      e2|   NULL|  purchase|  20.0|2026-01-01 10:05:00|2026-01-01 10:05:00|   missing_user|   fals

In [6]:
spark.stop()